# Mount Google Drive




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Connect to Google Earth Engine

In [ ]:
import ee
from google.colab import auth

auth.authenticate_user()
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='geovisionpakistan')

print("GEE connected successfully")

GEE connected successfully


#  Download Sentinel-2

In [ ]:
import time

pakistan = ee.Geometry.Rectangle([60.87, 23.63, 77.84, 37.09])

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(pakistan)
        .filterDate("2024-01-01", "2024-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
        .median()
        .select(["B2","B3","B4","B8","B11","B12"]))

task_s2 = ee.batch.Export.image.toDrive(
    image       = s2,
    description = "sentinel2_pakistan",
    region      = pakistan,
    scale       = 100,
    crs         = "EPSG:32642",
    maxPixels   = 1e13,
    fileFormat  = "GeoTIFF",
    folder      = "GeoVision/exports"
)

task_s2.start()
print("Sentinel-2 export started")
print("Task ID:", task_s2.id)

Sentinel-2 export started
Task ID: BNXVYX7MTTNR6CO2NSGRXA4Z


# Download WorldCover Labels

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

remapped = worldcover.remap(
    [10,  20,  30,  40,  50,  60,  80,  90],
    [ 2,   2,   2,   1,   0,   4,   3,   6],
    defaultValue=5
).rename("label").toByte()

task_wc = ee.batch.Export.image.toDrive(
    image       = remapped,
    description = "worldcover_labels",
    region      = pakistan,
    scale       = 100,
    crs         = "EPSG:32642",
    maxPixels   = 1e13,
    fileFormat  = "GeoTIFF",
    folder      = "GeoVision/exports"
)

task_wc.start()
print("WorldCover export started")
print("Task ID:", task_wc.id)

WorldCover export started
Task ID: BERFFZ4PA7UWGDMZ2GUU4XWY


# Monitor Both Downloads Together

In [ ]:
tasks = {
    "Sentinel-2"  : task_s2.id,
    "WorldCover"  : task_wc.id
}

start = time.time()

while True:
    all_done = True
    elapsed  = time.time() - start
    mins, secs = divmod(int(elapsed), 60)

    for name, task_id in tasks.items():
        task_obj = [t for t in ee.batch.Task.list()
                    if t.id == task_id][0]
        state    = task_obj.status()['state']
        print(f"[{mins:02d}:{secs:02d}] {name}: {state}")

        if state not in ('COMPLETED','FAILED','CANCELLED'):
            all_done = False

    if all_done:
        break

    print("---")
    time.sleep(60)

print("Both exports finished")

[00:00] Sentinel-2: READY
[00:00] WorldCover: READY
---
[01:00] Sentinel-2: READY
[01:00] WorldCover: READY
---
[02:01] Sentinel-2: READY
[02:01] WorldCover: READY
---
[03:02] Sentinel-2: READY
[03:02] WorldCover: READY
---
[04:02] Sentinel-2: READY
[04:02] WorldCover: READY
---
[05:03] Sentinel-2: READY
[05:03] WorldCover: READY
---
[06:04] Sentinel-2: READY
[06:04] WorldCover: READY
---
[07:04] Sentinel-2: READY
[07:04] WorldCover: READY
---
[08:05] Sentinel-2: READY
[08:05] WorldCover: READY
---
[09:07] Sentinel-2: READY
[09:07] WorldCover: READY
---
[10:08] Sentinel-2: READY
[10:08] WorldCover: READY
---
[11:08] Sentinel-2: READY
[11:08] WorldCover: READY
---
[12:09] Sentinel-2: READY
[12:09] WorldCover: READY
---
[13:10] Sentinel-2: READY
[13:10] WorldCover: READY
---
[14:11] Sentinel-2: READY
[14:11] WorldCover: READY
---
[15:12] Sentinel-2: READY
[15:12] WorldCover: READY
---
[16:12] Sentinel-2: READY
[16:12] WorldCover: READY
---
[17:13] Sentinel-2: READY
[17:13] WorldCover: RE

#  Install rasterio:

In [ ]:
!pip install rasterio -q
import rasterio
import numpy as np
import os

print("rasterio version:", rasterio.__version__)

rasterio version: 1.5.0


# Process One Tile At A Time

In [ ]:
!pip install scikit-image -q
print("Ready")

Ready


In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import os, time, shutil

# Paths
exports   = "/content/drive/MyDrive/GeoVision exports"
tiles_dir = "/content/drive/MyDrive/GeoVision exports/tiles"

os.makedirs(f"{tiles_dir}/images", exist_ok=True)
os.makedirs(f"{tiles_dir}/masks",  exist_ok=True)

# Your 4 Sentinel-2 tiles
s2_files = [
    f"{exports}/sentinel2_pakistan-0000000000-0000000000.tif",
    f"{exports}/sentinel2_pakistan-0000000000-0000009472.tif",
    f"{exports}/sentinel2_pakistan-0000009472-0000000000.tif",
    f"{exports}/sentinel2_pakistan-0000009472-0000009472.tif",
]

# WorldCover stays on Drive - read directly
wc_path = f"{exports}/worldcover_labels.tif"

TILE_SIZE  = 224
STRIDE     = 112
tile_count = 0
skip_count = 0

start = time.time()

# Process one S2 tile at a time
for s2_path in s2_files:

    print(f"\nProcessing: {os.path.basename(s2_path)}")

    with rasterio.open(s2_path) as s2_src:

        H = s2_src.height
        W = s2_src.width

        print(f"Size: {H} rows x {W} cols")

        # Get the geographic bounds of this S2 tile
        bounds = s2_src.bounds

        # Open WorldCover and find matching region
        with rasterio.open(wc_path) as wc_src:

            # Convert S2 bounds to WorldCover pixel window
            wc_window = rasterio.windows.from_bounds(
                bounds.left, bounds.bottom,
                bounds.right, bounds.top,
                wc_src.transform
            )

            # Read matching WorldCover region
            wc_data = wc_src.read(1, window=wc_window)

            # Resize to match S2 dimensions if needed
            if wc_data.shape != (H, W):
                from skimage.transform import resize
                wc_data = resize(
                    wc_data, (H, W),
                    order=0,
                    preserve_range=True,
                    anti_aliasing=False
                ).astype(np.uint8)

        # Now tile this S2 file
        for row in range(0, H - TILE_SIZE, STRIDE):
            for col in range(0, W - TILE_SIZE, STRIDE):

                window = Window(col, row, TILE_SIZE, TILE_SIZE)

                # Read image tile
                img_tile = s2_src.read(window=window)

                # Cut matching mask tile
                msk_tile = wc_data[row:row+TILE_SIZE,
                                   col:col+TILE_SIZE]

                # Validity check
                total  = img_tile.size
                zeros  = np.sum(img_tile == 0)
                mean_v = np.mean(img_tile)

                if (zeros/total > 0.30) or (mean_v > 8000):
                    skip_count += 1
                    continue

                # Save matched pair
                np.save(
                    f"{tiles_dir}/images/tile_{tile_count:06d}.npy",
                    img_tile
                )
                np.save(
                    f"{tiles_dir}/masks/tile_{tile_count:06d}.npy",
                    msk_tile
                )

                tile_count += 1

        elapsed = time.time() - start
        mins, secs = divmod(int(elapsed), 60)
        print(f"[{mins:02d}:{secs:02d}] Running total: {tile_count} saved, {skip_count} skipped")

print(f"\nFINISHED")
print(f"Total saved : {tile_count} tiles")
print(f"Total skipped: {skip_count} tiles")


Processing: sentinel2_pakistan-0000000000-0000000000.tif
Size: 9472 rows x 9472 cols
[05:53] Running total: 6889 saved, 0 skipped

Processing: sentinel2_pakistan-0000000000-0000009472.tif
Size: 9472 rows x 7882 cols
[10:48] Running total: 12616 saved, 0 skipped

Processing: sentinel2_pakistan-0000009472-0000000000.tif
Size: 5575 rows x 9472 cols
[14:20] Running total: 16600 saved, 0 skipped

Processing: sentinel2_pakistan-0000009472-0000009472.tif
Size: 5575 rows x 7882 cols
[17:19] Running total: 19912 saved, 0 skipped

FINISHED
Total saved : 19912 tiles
Total skipped: 0 tiles


# Sanity Check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

tiles_dir = "/content/drive/MyDrive/GeoVision exports/tiles"

# Load 3 random tiles and check them
import random
all_tiles = os.listdir(f"{tiles_dir}/images")
sample    = random.sample(all_tiles, 3)

for filename in sample:

    img  = np.load(f"{tiles_dir}/images/{filename}")
    mask = np.load(f"{tiles_dir}/masks/{filename}")

    print(f"File     : {filename}")
    print(f"Img shape: {img.shape}")
    print(f"Img min  : {img.min()}  max: {img.max()}")
    print(f"Mask shape: {mask.shape}")
    print(f"Mask unique values: {np.unique(mask)}")
    print("---")

File     : tile_004368.npy
Img shape: (6, 224, 224)
Img min  : 376.2  max: 5457.25
Mask shape: (224, 224)
Mask unique values: [1 2 3 4]
---
File     : tile_013354.npy
Img shape: (6, 224, 224)
Img min  : 206.5  max: 4932.0
Mask shape: (224, 224)
Mask unique values: [0 1 2 3 4 6]
---
File     : tile_016008.npy
Img shape: (6, 224, 224)
Img min  : 0.0  max: 4175.5
Mask shape: (224, 224)
Mask unique values: [0 1 2 3 4 6]
---


# The Train/Val/Test Split

In [ ]:
import os, json, random

tiles_dir = "/content/drive/MyDrive/GeoVision exports/tiles"

# Get all tile paths
all_tiles = sorted([
    f"{tiles_dir}/images/{f}"
    for f in os.listdir(f"{tiles_dir}/images")
])

# Shuffle with fixed seed for reproducibility
random.seed(42)
random.shuffle(all_tiles)

# 70/10/20 split
n         = len(all_tiles)
train_end = int(0.70 * n)
val_end   = int(0.80 * n)

splits = {
    "train": all_tiles[:train_end],
    "val":   all_tiles[train_end:val_end],
    "test":  all_tiles[val_end:]
}

# Save split file
split_path = f"{tiles_dir}/splits.json"
with open(split_path, "w") as f:
    json.dump(splits, f)

print(f"Train : {len(splits['train'])} tiles")
print(f"Val   : {len(splits['val'])}   tiles")
print(f"Test  : {len(splits['test'])}  tiles")
print(f"Saved : {split_path}")

Train : 13938 tiles
Val   : 1991   tiles
Test  : 3983  tiles
Saved : /content/drive/MyDrive/GeoVision exports/tiles/splits.json
